In [1]:
import unicodedata

def strip_diacritics(s: str) -> str:
    s = s.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn")

tests = [
    ("Tôi yêu tiếng Việt", "Toi yeu tieng Viet"),
    ("Cô ấy có một con mèo", "Co ay co mot con meo"),
    ("Trường đại học ở Hà Nội", "Truong dai hoc o Ha Noi"),
    ("Đường đi rất xa", "Duong di rat xa"),
]
for inp, exp in tests:
    got = strip_diacritics(inp)
    print(repr(inp), "->", repr(got), "OK" if got == exp else f"SAI (muon {exp!r})")

'Tôi yêu tiếng Việt' -> 'Toi yeu tieng Viet' OK
'Cô ấy có một con mèo' -> 'Co ay co mot con meo' OK
'Trường đại học ở Hà Nội' -> 'Truong dai hoc o Ha Noi' OK
'Đường đi rất xa' -> 'Duong di rat xa' OK


In [2]:
corpus = [
    "Tôi yêu tiếng Việt",
    "Cô ấy có một con mèo",
    "Trường đại học ở Hà Nội",
    "Đường đi rất xa",
    "Hôm nay trời nắng đẹp",
    "Chúng tôi học môn xử lý ngôn ngữ tự nhiên",
    "Mẹ tôi nấu cơm rất ngon",
    "Anh ấy đi làm bằng xe máy",
    "Cuốn sách này rất hay",
    "Tôi muốn khôi phục dấu tiếng Việt",
    "Hà Nội là thủ đô của Việt Nam",
    "Sinh viên làm bài tập trên Kaggle",
]
pairs = [(strip_diacritics(s), s) for s in corpus]
train, test = pairs[:9], pairs[9:]
print(f"corpus: {len(corpus)} cau | train {len(train)} | test {len(test)}")

corpus: 12 cau | train 9 | test 3


In [3]:
# Cell 3 — Baseline từ điển trên corpus mẫu 12 câu (demo).
# Số accuracy ở đây chỉ để minh họa cách chạy — KHÔNG đưa vào báo cáo
# (số liệu báo cáo đo trên test 10.000 câu ở Cell 6/Cell 8).
import collections

freq = collections.defaultdict(collections.Counter)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        freq[b][a] += 1
mapping = {b: c.most_common(1)[0][0] for b, c in freq.items()}
print("tu dien hoc duoc:", len(mapping), "tu khong dau")

def restore(bare: str) -> str:
    return " ".join(mapping.get(w, w) for w in bare.split())

tot = cor = 0
for bare, acc in test:
    pred = restore(bare)
    w_cor = sum(p == a for p, a in zip(pred.split(), acc.split()))
    tot += len(acc.split()); cor += w_cor
    print(f"  IN : {bare}\n  OUT: {pred}\n  GOLD: {acc}  ({w_cor}/{len(acc.split())} tu dung)")
print(f"\nword accuracy tren test mini: {cor}/{tot} = {cor/tot:.1%}")

tu dien hoc duoc: 45 tu khong dau
  IN : Toi muon khoi phuc dau tieng Viet
  OUT: Tôi muon khoi phuc dau tiếng Việt
  GOLD: Tôi muốn khôi phục dấu tiếng Việt  (3/7 tu dung)
  IN : Ha Noi la thu do cua Viet Nam
  OUT: Hà Nội la thu do cua Việt Nam
  GOLD: Hà Nội là thủ đô của Việt Nam  (4/8 tu dung)
  IN : Sinh vien lam bai tap tren Kaggle
  OUT: Sinh vien làm bai tap tren Kaggle
  GOLD: Sinh viên làm bài tập trên Kaggle  (3/7 tu dung)

word accuracy tren test mini: 10/22 = 45.5%


In [4]:
from datasets import load_dataset
import re

N_ARTICLES = 10000
TARGET_SENTENCES = 100000

ds = load_dataset("wikimedia/wikipedia", "20231101.vi",
                  split="train", streaming=True)

sentences, seen = [], set()
n_read = 0
for ex in ds:
    n_read += 1
    if n_read > N_ARTICLES:
        break
    text = ex["text"].replace("\n", " ")
    for s in re.split(r"(?<=[.!?…])\s+(?=[A-ZÀ-ỸĐ])", text):
        s = re.sub(r"\s+", " ", s).strip()
        w = s.split()
        if not (5 <= len(w) <= 40):
            continue
        if "http" in s or s.count("=") > 2:
            continue
        if sum(c.isdigit() for c in s) / max(len(s), 1) > 0.15:
            continue
        if s in seen:
            continue
        seen.add(s)
        sentences.append(s)
        if len(sentences) >= TARGET_SENTENCES:
            break
    if len(sentences) >= TARGET_SENTENCES:
        break

print(f"bai viet da doc: {n_read} | cau giu lai: {len(sentences)}")
for s in sentences[:5]:
    print(" -", s)

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

bai viet da doc: 1501 | cau giu lai: 100000
 - Internet Society hay ISOC là một tổ chức quốc tế hoạt động phi lợi nhuận, phi chính phủ và bao gồm các thành viên có trình độ chuyên ngành.
 - Tổ chức này chú trọng đến: tiêu chuẩn, giáo dục và các vấn đề về chính sách.
 - Với trên 145 tổ chức thành viên và 65.000 thành viên cá nhân, ISOC bao gồm những con người cụ thể trong cộng đồng Internet.
 - Mọi chi tiết có thể tìm thấy tại website của ISOC.
 - Internet Society nằm ở gần thủ đô Washington, DC, Hoa Kỳ và Geneva, Thụy Sĩ.


In [5]:
import collections, random

random.seed(42)
# Chuẩn hóa câu về NFC: Wikipedia có chỗ lưu dấu dạng tách rời (NFD,
# vd "ê" = "e" + dấu mũ tách riêng) khiến từ có dấu và từ không dấu
# lệch độ dài. NFC gộp mỗi ký tự có dấu thành 1 ký tự duy nhất.
pairs = [(strip_diacritics(s), unicodedata.normalize("NFC", s))
         for s in sentences]
random.shuffle(pairs)
n = len(pairs)
n_train, n_val = int(n * 0.8), int(n * 0.1)
train = pairs[:n_train]
val = pairs[n_train:n_train + n_val]
test = pairs[n_train + n_val:]
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

words = [w for _, acc in pairs for w in acc.split()]
n_words = len(words)
n_accented = sum(1 for w in words if strip_diacritics(w) != w)
print(f"tong so tu: {n_words} | tu co dau: {n_accented} "
      f"({n_accented / n_words:.1%})")

variants = collections.defaultdict(collections.Counter)
for bare, acc in pairs:
    for b, a in zip(bare.split(), acc.split()):
        variants[b][a] += 1
amb = {b: c for b, c in variants.items() if len(c) >= 2}
print(f"tu khong dau co >=2 dang co dau: {len(amb)} / {len(variants)}")
for b, c in sorted(amb.items(), key=lambda kv: -sum(kv[1].values()))[:10]:
    print(f"  {b!r}: {dict(c.most_common(5))}")

# Tập từ đa nghĩa DÙNG CHO METRIC (Cell 6/Cell 8): chỉ tính trên train —
# định nghĩa chung với Notebook 2/3. Tính trên toàn bộ pairs (kể cả test)
# sẽ để test rò rỉ vào định nghĩa metric.
forms_train = collections.defaultdict(set)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        forms_train[b].add(a)
amb_set = {b for b, v in forms_train.items() if len(v) >= 2}
print(f"tu da nghia tren train (dung cho metric): {len(amb_set)}")

train 80000 | val 10000 | test 10000
tong so tu: 2387449 | tu co dau: 1933227 (81.0%)
tu khong dau co >=2 dang co dau: 6078 / 78862
  'va': {'và': 40197, 'va': 188, 'vá': 20, 'vã': 14, 'vạ': 12}
  'cua': {'của': 36241, 'cửa': 547, 'cua': 37, 'cưa': 20, 'cựa': 2}
  'la': {'là': 35547, 'lá': 336, 'la': 186, 'lạ': 64, 'lả': 3}
  'co': {'có': 26628, 'cơ': 3681, 'cổ': 1399, 'cố': 692, 'cô': 258}
  'cac': {'các': 28681, 'càc': 1, 'cac': 1, 'cạc': 1}
  'duoc': {'được': 25853, 'dược': 83, 'đuốc': 2, 'đước': 2, 'dựoc': 1}
  'trong': {'trong': 20899, 'trọng': 2035, 'trồng': 319, 'trông': 103, 'trống': 99}
  'mot': {'một': 22071, 'mốt': 5, 'mọt': 3, 'từ': 1}
  'tu': {'từ': 10844, 'tự': 3720, 'tử': 3426, 'tư': 1405, 'tụ': 307}
  'do': {'độ': 4933, 'đó': 4916, 'do': 3937, 'đô': 1284, 'đồ': 1057}


In [6]:
freq = collections.defaultdict(collections.Counter)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        freq[b][a] += 1
mapping = {b: c.most_common(1)[0][0] for b, c in freq.items()}
print("tu dien hoc duoc:", len(mapping), "tu khong dau")

def restore(bare: str) -> str:
    return " ".join(mapping.get(w, w) for w in bare.split())

tot = cor = sent_tot = sent_cor = amb_tot = amb_cor = 0
for bare, acc in test:
    pred = restore(bare)
    pw, aw = pred.split(), acc.split()
    sent_tot += 1
    sent_cor += (pred == acc)
    for p, a, b in zip(pw, aw, bare.split()):
        tot += 1
        cor += (p == a)
        if b in amb_set:
            amb_tot += 1
            amb_cor += (p == a)

print(f"word accuracy: {cor}/{tot} = {cor / tot:.1%}")
print(f"sentence accuracy: {sent_cor}/{sent_tot} = {sent_cor / sent_tot:.1%}")
if amb_tot:
    print(f"accuracy tren tu da nghia: {amb_cor}/{amb_tot} = "
          f"{amb_cor / amb_tot:.1%}")
for bare, acc in test[:3]:
    print(f"  IN : {bare}\n  OUT: {restore(bare)}\n  GOLD: {acc}")

tu dien hoc duoc: 68067 tu khong dau
word accuracy: 185248/239189 = 77.4%
sentence accuracy: 261/10000 = 2.6%
accuracy tren tu da nghia: 156257/209279 = 74.7%
  IN : Moi de doa cua cac tran dong dat lon dong vai tro lon trong viec phat trien co so ha tang cua thanh pho.
  OUT: Mỗi để dọa của các trận động đất lớn động vài trở lớn trong việc phát triển có số hạ tăng của thành phố.
  GOLD: Mối đe dọa của các trận động đất lớn đóng vai trò lớn trong việc phát triển cơ sở hạ tầng của thành phố.
  IN : Trong thoi gian nong nhat trong nam (thang 3 - thang 4), nhiet do thuong dat toi tro len, ngoai tru vung duyen hai, noi gio bien co nhiet do diu buoi chieu.
  OUT: Trong thời gian nông nhất trong năm (tháng 3 - tháng 4), nhiệt đó thường đất tới trở lên, ngoài trụ vùng duyên hai, nơi giờ biến có nhiệt đó dịu buổi chiều.
  GOLD: Trong thời gian nóng nhất trong năm (tháng 3 - tháng 4), nhiệt độ thường đạt tới trở lên, ngoại trừ vùng duyên hải, nơi gió biển có nhiệt độ dịu buổi chiều.
  IN : Va n

In [7]:
# Cell 7 — Lưu 3 tập train/val/test ra file CSV (để kiểm tra / dùng lại
# bên ngoài notebook). Lưu ý: mỗi notebook Kaggle có /kaggle/working
# riêng nên Notebook 2/3 không đọc được file này — chúng tái tạo đúng
# tập dữ liệu bằng cách chạy lại cùng code + seed 42 (Cell 1), và kiểm
# chứng bằng assert tổng số từ = 2.387.449.
import csv

def save_pairs(pairs, path):
    # Ghi danh sách cặp (không dấu, có dấu) ra file CSV
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["bare", "acc"])
        w.writerows(pairs)

save_pairs(train, "/kaggle/working/train.csv")
save_pairs(val, "/kaggle/working/val.csv")
save_pairs(test, "/kaggle/working/test.csv")
print("da luu train.csv, val.csv, test.csv")

da luu train.csv, val.csv, test.csv


In [8]:
# Cell 8 — Baseline n-gram (bigram + giải mã Viterbi)
# Ý tưởng: từ điển (Cell 6) chọn dạng có dấu phổ biến nhất mà không nhìn
# ngữ cảnh, nên sai ở từ đa nghĩa (vd "co" -> có/cô/cỏ). Bigram nhìn thêm
# từ đứng trước: chọn dạng có dấu sao cho cặp từ liền kề "nghe quen"
# nhất theo thống kê trên tập train. Viterbi tìm dãy dạng có dấu tốt nhất
# cho cả câu thay vì chọn tham lam từng từ một.
import math
from collections import Counter, defaultdict

# Đếm unigram và bigram trên câu có dấu của tập train
uni = Counter()   # số lần mỗi từ có dấu xuất hiện
bi = Counter()    # số lần mỗi cặp từ có dấu liền kề xuất hiện
for _, acc in train:
    toks = ["<s>"] + acc.split()
    uni.update(toks)
    bi.update(zip(toks, toks[1:]))

V = len(uni)  # số từ có dấu khác nhau (dùng làm mịn Laplace)
ALPHA = 0.1   # hệ số làm mịn cộng thêm

def log_prob(prev, word):
    # log P(word | prev) với làm mịn add-alpha
    return math.log((bi[(prev, word)] + ALPHA) / (uni[prev] + ALPHA * V))

# Các dạng có dấu ứng viên cho mỗi dạng không dấu (học từ train)
cands = defaultdict(set)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        cands[b].add(a)

def restore_bigram(bare):
    # Giải mã Viterbi: tìm dãy dạng có dấu có tổng log-prob lớn nhất
    words = bare.split()
    # dp[i][dạng] = điểm tốt nhất tới vị trí i khi từ i chọn dạng đó
    dp = [{"<s>": 0.0}]
    back = []
    for w in words:
        opts = cands.get(w, {w})  # từ chưa thấy trong train: giữ nguyên
        prev_dp = dp[-1]
        cur, bp = {}, {}
        for o in opts:
            best_prev, best_score = None, float("-inf")
            for p, s in prev_dp.items():
                sc = s + log_prob(p, o)
                if sc > best_score:
                    best_prev, best_score = p, sc
            cur[o], bp[o] = best_score, best_prev
        dp.append(cur)
        back.append(bp)
    # Truy ngược để lấy dãy từ tốt nhất
    last = max(dp[-1], key=dp[-1].get)
    out = [last]
    for bp in reversed(back):
        last = bp[last]
        out.append(last)
    return " ".join(reversed(out[:-1]))  # bỏ token <s> ở đầu

# Đo 3 metric giống Cell 6, trên cùng tập test
tot = cor = sent_tot = sent_cor = amb_tot = amb_cor = 0
for bare, acc in test:
    pred = restore_bigram(bare)
    pw, aw = pred.split(), acc.split()
    sent_tot += 1
    sent_cor += (pred == acc)
    for p, a, b in zip(pw, aw, bare.split()):
        tot += 1
        cor += (p == a)
        if b in amb_set:
            amb_tot += 1
            amb_cor += (p == a)

print(f"bigram — word accuracy: {cor}/{tot} = {cor / tot:.1%}")
print(f"bigram — sentence accuracy: {sent_cor}/{sent_tot} = {sent_cor / sent_tot:.1%}")
if amb_tot:
    print(f"bigram — accuracy tren tu da nghia: {amb_cor}/{amb_tot} = "
          f"{amb_cor / amb_tot:.1%}")
for bare, acc in test[:3]:
    print(f"  IN : {bare}\n  OUT: {restore_bigram(bare)}\n  GOLD: {acc}")

bigram — word accuracy: 219949/239189 = 92.0%
bigram — sentence accuracy: 3104/10000 = 31.0%
bigram — accuracy tren tu da nghia: 190958/209279 = 91.2%
  IN : Moi de doa cua cac tran dong dat lon dong vai tro lon trong viec phat trien co so ha tang cua thanh pho.
  OUT: Mối đe dọa của các trận động đất lớn đóng vai trò lớn trong việc phát triển cơ sở hạ tầng của thành phố.
  GOLD: Mối đe dọa của các trận động đất lớn đóng vai trò lớn trong việc phát triển cơ sở hạ tầng của thành phố.
  IN : Trong thoi gian nong nhat trong nam (thang 3 - thang 4), nhiet do thuong dat toi tro len, ngoai tru vung duyen hai, noi gio bien co nhiet do diu buoi chieu.
  OUT: Trong thời gian nóng nhất trong năm (tháng 3 - tháng 4), nhiệt độ thường đạt tới trở lên, ngoại trừ vùng duyên hải, nổi gió biến và nhiệt độ dìu buổi chiều.
  GOLD: Trong thời gian nóng nhất trong năm (tháng 3 - tháng 4), nhiệt độ thường đạt tới trở lên, ngoại trừ vùng duyên hải, nơi gió biển có nhiệt độ dịu buổi chiều.
  IN : Va nguoc lai